# Predicting House Prices in California with `LinearRegression()`

In this lab you will start inspect, analyze, visualize house price data from different districts in California, US. After having performed analysis, EDA and some feature engineering, you will build your own `LinearRegression()`  with `SkLearn`. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

# Part 1 - Inspection and Cleaning


#### Import and Inspect your data

Read the `housing.csv` file and make use of some methods to understand your data better. Below is an explanation of the features you are going to work with:

1. **longitude:**  geographical coordinate, east to west position of district
2. **latitude:**  geographical coordinate, north to south position of district
3. **housing_median_age:** the median age of houses in district
4. **total_rooms** Sum of all rooms in district
5. **total_bedrooms** Sum of all bedrooms in district
6. **population:** total population in district
7. **households:** total households in district
8. **median_income:** median household income in district 
9. **median_house_value:** median house value in district
10. **ocean_proximity:** District´s proximity to the ocean

In [ ]:
file_path = Path("housing.csv")

if not file_path.exists():
    file_path = Path("data") / "housing.csv"

housing = pd.read_csv(file_path)

In [ ]:
housing.head()

In [ ]:
housing.info()

In [ ]:
housing.describe()

#### Histograms
Make histograms of all your numeric columns in order to get a good understanding of the distribution of your data points. What do you see?

In [ ]:
housing.hist(bins=50, figsize=(14, 10))
plt.tight_layout()
plt.show()

# Answer: Most numeric columns are right-skewed. The median house value also has a clear cap at 500001.

#### Let's create some features a tidy up our data

1. Locate your NaN values and make a decision on how to handle them. Drop, fill with mean, or something else, it is entirely up to you. 

In [ ]:
housing.isna().sum()

In [ ]:
housing["total_bedrooms"] = housing["total_bedrooms"].fillna(housing["total_bedrooms"].median())
housing.isna().sum()

2. Create three new columns by using simple arithmetic operations. Create one column with "rooms per household", one with "population per household",  and one with "bedrooms per room".

In [ ]:
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]

In [ ]:
housing["population_per_household"] = housing["population"] / housing["households"]

In [ ]:
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing[["rooms_per_household", "population_per_household", "bedrooms_per_room"]].head()

3. If you check the largest and smallest values of your "rooms per houshold column" you will see two outliers and two values that are just wrong. Drop the four values by index.

In [ ]:
housing["rooms_per_household"].nlargest(2)

In [ ]:
housing["rooms_per_household"].nsmallest(2)

In [ ]:
outlier_index = housing["rooms_per_household"].nlargest(2).index.tolist() + housing["rooms_per_household"].nsmallest(2).index.tolist()
outlier_index

In [ ]:
housing = housing.drop(index=outlier_index)
housing.shape

# Part 2 - Exploratory Data Analysis



#### Let's find out what factors have an influence on our predicting variable

1. Let's check out the distribution of our "median house value". Visualize your results with 100 bins.

In [ ]:
housing["median_house_value"].hist(bins=100, figsize=(8, 5))
plt.xlabel("Median house value")
plt.ylabel("Count")
plt.show()

2. Check out what variables correlates the most with "median house value"

In [ ]:
corr_matrix = housing.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

3. Let's check out the distribution of the column that has the highest correlation to "median house value". Visualize your results with 100 bins.

In [ ]:
housing["median_income"].hist(bins=100, figsize=(8, 5))
plt.xlabel("Median income")
plt.ylabel("Count")
plt.show()

4. Visualize the "median house value" and "median income" in a jointplot (kind="reg"). What do you see?

In [ ]:
sns.jointplot(data=housing, x="median_income", y="median_house_value", kind="reg", height=7)
plt.show()

# Answer: There is a clear positive relationship. Higher income usually means higher house value.

5. Make the same visualization as in the above, but, cahnge the kind parameter to "kde". What extra information does this type of visualization convey, that the one in the above does not?

In [ ]:
eda_sample = housing.sample(3000, random_state=42)
sns.jointplot(data=eda_sample, x="median_income", y="median_house_value", kind="kde")
plt.show()

# The KDE plot shows where most observations are concentrated, not only the general trend.

#### Let's get schwifty with some EDA

1. Create a new categorical column from the "median income" with the following quartiles `[0, 0.25, 0.5, 0.75, 0.95, 1]` and label them like this `["Low", "Below_Average", "Above_Average", "High", "Very High"]` and name the column "income_cat"

In [ ]:
housing["income_cat"] = pd.qcut(
    housing["median_income"],
    q=[0, 0.25, 0.5, 0.75, 0.95, 1],
    labels=["Low", "Below_Average", "Above_Average", "High", "Very High"]
)

housing[["median_income", "income_cat"]].head()

2. Using the Seaborn library, plot the count of your new column and set the `hue` to "ocean_proximity". What interesting things can you see?

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=housing, x="income_cat", hue="ocean_proximity")
plt.xticks(rotation=30)
plt.show()

# Answer: INLAND has many low income areas, while areas closer to the ocean appear more in higher income groups.

3. Create two barplots where you set "y="median_house_value" on both, and the x is first "income cat" and then "ocean_proximity". How does these two graphs complement what you saw in the graph in your previous question?

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=housing, x="income_cat", y="median_house_value", color="C0")
plt.xticks(rotation=30)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=housing, x="ocean_proximity", y="median_house_value", color="C0")
plt.xticks(rotation=30)
plt.show()

# Answer: The barplots show that price increases with income, and houses near the ocean/bay are more expensive than inland houses.

4. Create a pivoted dataframe where you have the values of the "income cat" column as indices and the values of the "ocean_proximity" column as columns. Also drop the "ISLAND" column that you'll get.

In [ ]:
pivot_df = housing.pivot_table(
    index="income_cat",
    columns="ocean_proximity",
    values="median_house_value",
    aggfunc="mean",
    observed=False
)

In [ ]:
pivot_df = pivot_df.drop(columns="ISLAND")
pivot_df

5. Turn your pivoted dataframe into a heatmap. The heatmap should have annotations in integer format.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, annot=True, fmt=".0f")
plt.show()

# Part 3 - Preparing your Data



#### Splitting, Preparing and Engineering some Features

1. Let's drop the "income_cat" column as it has served its purpose already. We don't need for our model as we already have "median income".
Not dropping "incom cat" will lead to multicolinearity.

In [ ]:
housing = housing.drop(columns=["income_cat"], errors="ignore")
housing.head()

2. Select your floating point columns and standardize your data by calculating the Z-score. You can apply the `stats.zscore()` method in a lambda function. Save your results to a variable called `z_scored`. 

In [ ]:
import scipy.stats as stats

In [ ]:
float_cols = housing.select_dtypes(include="float").columns
z_scored = housing[float_cols].apply(lambda x: stats.zscore(x))
z_scored.head()

3. Turn the only categorical columns into dummies. Be vary of the dummy trap, to avoid multicolinearity.

In [ ]:
dummies = pd.get_dummies(housing["ocean_proximity"], drop_first=True, dtype=int)
dummies.head()

4. Save our predicting variable to `y`.

In [ ]:
y = housing["median_house_value"]
y.head()

5. Concatenate `z_scored` and `dummies` and drop the predicting variable. Save to the varible `X`.

In [ ]:
X = pd.concat([z_scored, dummies], axis=1).drop(columns="median_house_value")
X.head()

# Part 4 - Machine Learning 




#### Train, Test, Split

1. Import `train_test_split` and split your data accordingly. Choose an appropriate test size.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#### Building and Training our Model

2. Build, fit and train a `LinearRegression` model. 

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

3. In a scatterplot, visualize the y_train on your x-axis and your predictions on the y-axis. How does your training predictions look? 

In [ ]:
train_predictions = model.predict(X_train)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_train, y=train_predictions, alpha=0.4)
plt.xlabel("Real values")
plt.ylabel("Predicted values")
plt.show()

# Answer: The predictions follow the general direction, but they are still quite spread out.

4. From the sklearn metrics module, print the mean_squared_error and R^2-score. What does the metrics tell us?

In [ ]:
from sklearn import metrics

In [ ]:
train_mse = metrics.mean_squared_error(y_train, train_predictions)
train_mse

In [ ]:
train_r2 = metrics.r2_score(y_train, train_predictions)
train_r2

# Answer: The model explains part of the price variation, but it is not perfect.

#### Final Predictions

1. Now you are ready to make prediction on the test data. Do that and visualize your results in a new scatterplot.

In [ ]:
test_predictions = model.predict(X_test)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=test_predictions, alpha=0.4)
plt.xlabel("Real values")
plt.ylabel("Predicted values")
plt.show()

2. Print the mean_squared_error and R^2-score again. What has happened?

In [ ]:
test_mse = metrics.mean_squared_error(y_test, test_predictions)
test_mse

In [ ]:
test_r2 = metrics.r2_score(y_test, test_predictions)
test_r2

# Answer: The test score is close to the train score, so the model is not overfitting too much.

3. There is another metric called Root mean squared error, Which is the square root of the MSE. Calculate the RMSE.

In [ ]:
rmse = np.sqrt(test_mse)
rmse

# Bonus Questions 1

1. Create a dataframe with two columns, one consisting of the y_test and one of your model's predictions.

In [ ]:
results_df = pd.DataFrame({
    "actual": y_test,
    "predicted": test_predictions
})

results_df.head()

2. Make a series of of your new dataframe, by calculating the predicted error in absolut numbers. Save this series to variable name `absolute_errors`.

In [ ]:
absolute_errors = abs(results_df["actual"] - results_df["predicted"])

print("Mean absolute error:", absolute_errors.mean())
absolute_errors.head()

3. If you take the mean of your series, you will get the mean absolute errors, which is another metric for Linear Regressions.

# Bonus Question 2 - Build a Random Forest Regressor

1. Build, fit and train a `RandomForestRegressor` model. Do this by following the same staps that you followed when building your `LinearRegression`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
rf_train_predictions = rf_model.predict(X_train)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_train, y=rf_train_predictions, alpha=0.4)
plt.xlabel("Real values")
plt.ylabel("Random Forest train predictions")
plt.show()

In [ ]:
rf_train_mse = metrics.mean_squared_error(y_train, rf_train_predictions)
rf_train_r2 = metrics.r2_score(y_train, rf_train_predictions)
rf_train_rmse = np.sqrt(rf_train_mse)

print("Random Forest train MSE:", rf_train_mse)
print("Random Forest train R2:", rf_train_r2)
print("Random Forest train RMSE:", rf_train_rmse)

2. Make prediction on the test data and evaluate you results.

In [ ]:
rf_test_predictions = rf_model.predict(X_test)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=y_test, y=rf_test_predictions, alpha=0.4)
plt.xlabel("Real values")
plt.ylabel("Random Forest test predictions")
plt.show()

In [ ]:
rf_test_mse = metrics.mean_squared_error(y_test, rf_test_predictions)
rf_test_r2 = metrics.r2_score(y_test, rf_test_predictions)

print("Random Forest test MSE:", rf_test_mse)
print("Random Forest test R2:", rf_test_r2)

In [ ]:
rf_test_rmse = np.sqrt(rf_test_mse)
rf_test_mae = metrics.mean_absolute_error(y_test, rf_test_predictions)

print("Random Forest test RMSE:", rf_test_rmse)
print("Random Forest test MAE:", rf_test_mae)

In [ ]:
print("The Random Forest performs better than the Linear Regression, but it may overfit a bit because the train score is much higher than the test score.")